In [24]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
%matplotlib inline

In [25]:
#read in all the words
words = open("names.txt", 'r').read().splitlines()
words[:10]


['emma',
 'olivia',
 'ava',
 'isabella',
 'sophia',
 'charlotte',
 'mia',
 'amelia',
 'harper',
 'evelyn']

In [26]:
len(words)

32033

In [27]:
#build the vocabulry of characers and mappings to/from integer
chars = sorted(list(set(''.join(words))))
stoi = {s:i+1 for i , s in enumerate(chars)}
stoi['.'] = 0
itos = {i:s for s, i in stoi.items()}
print(itos)

{1: 'a', 2: 'b', 3: 'c', 4: 'd', 5: 'e', 6: 'f', 7: 'g', 8: 'h', 9: 'i', 10: 'j', 11: 'k', 12: 'l', 13: 'm', 14: 'n', 15: 'o', 16: 'p', 17: 'q', 18: 'r', 19: 's', 20: 't', 21: 'u', 22: 'v', 23: 'w', 24: 'x', 25: 'y', 26: 'z', 0: '.'}


In [28]:
#build the data set

block_size = 3 # context length: how many characters do we take ro predict the next one?
X , Y = [], []
for w in words[:5]:
    print(w)
    context = [0]*block_size    
    for ch in w + '.':
        ix = stoi[ch]
        X.append(context)
        Y.append(ix)
        print(''.join(itos[i] for i in context), '--->', itos[ix])
        context = context[1:] + [ix]

X = torch.tensor(X)
Y = torch.tensor(Y)


emma
... ---> e
..e ---> m
.em ---> m
emm ---> a
mma ---> .
olivia
... ---> o
..o ---> l
.ol ---> i
oli ---> v
liv ---> i
ivi ---> a
via ---> .
ava
... ---> a
..a ---> v
.av ---> a
ava ---> .
isabella
... ---> i
..i ---> s
.is ---> a
isa ---> b
sab ---> e
abe ---> l
bel ---> l
ell ---> a
lla ---> .
sophia
... ---> s
..s ---> o
.so ---> p
sop ---> h
oph ---> i
phi ---> a
hia ---> .


In [29]:
X.shape, X.dtype, Y.shape , Y.dtype

(torch.Size([32, 3]), torch.int64, torch.Size([32]), torch.int64)

In [30]:
C = torch.randn((27,2))
C

tensor([[ 0.1988, -0.5703],
        [ 0.3479, -1.2927],
        [-0.6839, -0.3131],
        [ 2.2132, -0.2308],
        [-0.2099, -0.3702],
        [-0.3777, -0.7531],
        [ 1.3946, -0.7357],
        [-0.7039, -1.6860],
        [ 0.9244, -0.4375],
        [-0.1760, -1.0589],
        [ 0.7913, -0.0903],
        [ 1.0917,  2.0746],
        [ 0.3083, -0.5012],
        [-0.9283,  0.5658],
        [ 0.6377, -0.9447],
        [-0.2752,  1.0039],
        [-0.1622,  0.1000],
        [-0.5619, -0.7933],
        [-1.0024, -0.0361],
        [ 0.1502, -0.9845],
        [ 0.7012,  1.3398],
        [-0.5790, -0.0110],
        [ 0.1172, -0.1727],
        [ 0.6930, -0.6586],
        [-1.0832, -0.5567],
        [-0.1080, -1.4333],
        [-0.5455,  1.0756]])

In [31]:
C[torch.tensor([5,6,7])]
emb = C[X]

In [32]:
torch.cat(torch.unbind(emb, 1),1)

w1 = torch.randn((6,100))
b1 = torch.rand(100)
h = torch.tanh(emb.view(-1,6)@w1 + b1)
h

tensor([[-0.1248, -0.5157,  0.6727,  ...,  0.9467,  0.6699,  0.7498],
        [-0.3258, -0.5842,  0.7428,  ...,  0.9827,  0.8404,  0.9240],
        [ 0.3527,  0.1741,  0.9358,  ...,  0.7508,  0.5601,  0.9641],
        ...,
        [ 0.6525,  0.6869, -0.3619,  ..., -0.6422,  0.5054, -0.1083],
        [-0.6701,  0.1959,  0.9156,  ...,  0.9979,  0.9615,  0.9526],
        [ 0.1266, -0.8958,  0.5127,  ...,  0.6197,  0.9835,  0.4149]])

In [ ]:
w2 = torch.rand((100,27))
b2 = torch.rand(27)
logits = h @w2 + b2
Y 


tensor([ 5, 13, 13,  1,  0, 15, 12,  9, 22,  9,  1,  0,  1, 22,  1,  0,  9, 19,
         1,  2,  5, 12, 12,  1,  0, 19, 15, 16,  8,  9,  1,  0])

In [34]:

torch.arange(32)

tensor([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17,
        18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31])

In [38]:
parameters = [C, w1, b1, w2, b2]
for p in parameters:
    p.requires_grad = True
loss = F.cross_entropy(logits, Y)
for p in parameters:
    p.grad = None
loss.backward()
# for p in parameters:
#      p.data += -0.1*p.grad

RuntimeError: element 0 of tensors does not require grad and does not have a grad_fn